<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/03_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Tools an agent can actually use

New domain: a **customer support agent** that triages tickets against real orders and a refund
policy.

The model never sees your code. It sees a **name, a description, and a schema** — which makes a
tool's docstring one of the highest-leverage prompts you will write.

**New in this lesson:** `@tool`, docstrings as prompt surface, error messages as instructions

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-03-tools"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. The data

Six orders, eight tickets, one refund policy. Small on purpose: you should be able to read all
of it and judge the agent's answers yourself. If you cannot tell whether the agent is right,
you cannot tell whether your tools are working.

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

---

## 2. Three tools, written carefully

Read the docstrings below as if you were the model. They are all you would have.

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

In [ ]:
from deepagents import create_deep_agent

SUPPORT_PROMPT = (
    "You are a customer support agent for an office furniture retailer.\n"
    "Always look up the order before answering questions about it.\n"
    "Always check the refund policy before promising a refund, replacement, or exchange.\n"
    "Be concise and specific. Quote the relevant policy line."
)

support_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)
support_agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("support_agent")

Example prompts:
> Ticket T-1: the customer says order 1042 arrived with a cracked leg. What are their options?

> Order 1047 - the laptop stand wobbles. Can they get a refund?

Watch the tool calls in Studio. For the second prompt it should look up the order, notice it was
delivered 62 days ago, read the policy, and offer a **repair only**.

---

## 3. What the model actually received

Your Python is invisible. This is the entire contract.

In [ ]:
import json

for t in [lookup_order, search_tickets, get_refund_policy]:
    print(f"name:        {t.name}")
    print(f"description: {t.description}")
    print(f"schema:      {json.dumps(t.args)}")
    print("-" * 70)

The description **is** the docstring. The schema **is** the type hints. You did not write a
prompt for these tools — but you wrote a prompt for these tools.

This is why "just add a tool" so often fails: the tool works fine, and the model has no idea
when to reach for it.

---

## 4. The same capability, described badly

Identical logic. Different name, no useful description.

In [ ]:
from langchain_core.tools import tool


@tool
def get_data(x: str) -> str:
    """Gets data."""
    for order in ORDERS:
        if order["id"] == x:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago"
            )
    raise ValueError("not found")


@tool
def check(q: str) -> str:
    """Checks things."""
    return REFUND_POLICY


bad_agent = create_deep_agent(
    model=MODEL,
    tools=[get_data, check],
    system_prompt=SUPPORT_PROMPT,
)
bad_agent

In [ ]:
start_studio("bad_agent")

Example prompt:
> Ticket T-1: the customer says order 1042 arrived with a cracked leg. What are their options?

Send the **same** prompt you sent the first agent, and compare the tool calls.

`check` usually goes unused — the model has to guess whether "checks things" is relevant to
"what are this customer's options", and it often guesses no. It then answers from general
knowledge of how returns usually work, fluently and without a source.

A useful description answers **"when should I call this?"**, not "what does it do?".

---

## 5. Error messages are instructions

When a tool fails, its error text goes straight into the model's context. It is a message to a
reader who cannot see your code, cannot read your stack trace, and has to decide what to do next.

In [ ]:
@tool
def lookup_order_guides(order_id: str) -> str:
    """Look up an order by its 4-digit ID."""
    for order in ORDERS:
        if order["id"] == order_id:
            return str(order)
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits starting with 10 "
        f"(e.g. 1042). Ask the customer to check their confirmation email, or use "
        f"search_tickets to find the order from the ticket text."
    )


guided_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order_guides, search_tickets],
    system_prompt=SUPPORT_PROMPT,
)
guided_agent

In [ ]:
start_studio("guided_agent")

Example prompt:
> Ticket T-8 mentions order 9999 never arrived. Look it up and tell me what to do next.

Order 9999 does not exist. Because the error text says what to try instead, the agent usually
recovers — it searches the tickets or asks a sensible question. A tool that raised
`ValueError("not found")` here would typically apologise and stop.

**Write error strings for the model the way you would write them for a new colleague.**

---

## 6. Return-shape discipline

A tool that returns everything costs you on every future step, because tool results stay in the
message history (lesson 02).

The fix is the one the harness already uses: **write the bulk to a file, return the path plus a
one-line summary.** The agent reads the file only if it needs the detail.

In [ ]:
@tool
def get_ticket_history(order_id: str) -> str:
    """Fetch the full support history for an order.

    Writes the history to a file and returns the path plus a short summary. Read the file
    only if you need the detail.
    """
    body = "\n".join(
        f"[2026-0{i}-12] agent: reply about order {order_id} ... " + ("detail " * 60)
        for i in range(1, 9)
    )
    return (
        f"Wrote /tickets/{order_id}.md ({len(body)} chars, 8 messages). "
        f"Most recent: customer following up on a cracked desk leg. "
        f"Use read_file('/tickets/{order_id}.md') for the full text."
    )


print(get_ticket_history.invoke({"order_id": "1042"}))

Truncating would have **destroyed** the detail. Offloading **defers** it — the agent gets a cheap
summary now and can pay for the rest on purpose.

---

## 📌 Key takeaways

- The model sees a **name, a description, and a schema** — those three things are your entire tool interface.
- A good description answers *when should I call this?*, not *what does it do?*
- Badly described tools fail **silently**: you get a fluent, confident, unsourced answer.
- Error messages are instructions to a reader who cannot see your code. Say what to try next.
- Return the smallest thing that lets the agent decide its next step; write bulk to a file and return the path.
- You test a tool by watching the trace in Studio, not by checking its return value in isolation.

---

## ➡️ Next

**[04 · Tools II: MCP servers](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/04_mcp.ipynb)**

You wrote those tools. Next: tools you did **not** write and do not control — connecting an
agent to public MCP servers, and the context-budget and trust problems that come with them.